# Summary
Neste notebook nos iremos preparar nossa base presente em `00_dataset/00_raw_data`. Seguiremos os seguintes passos:
1. Daremos load em todos os arquivos `csv`;
2. Faremos os cruzamentos necessarios utilizando `duckdb`;
3. Usaremos os dados de `test` para inferencia `batch` e `real-time` posteriomente;
4. Noralizaremos a tabela ajustando os nomes das colunas;
5. Salvaremos no arquivo `00_dataset/01_prep_data`.

A partir desse arquivo seguiremos para a etapa `02` do projeto `Feature Engineering`. 

In [29]:
import re
import duckdb
import polars as pl
from pathlib import Path

In [51]:
FILE_PATH = Path('../00_dataset/00_raw_data/')

TRAIN_IDENTITY = FILE_PATH / 'train_identity.csv' 
TRAIN_TRANSACTION = FILE_PATH / 'train_transaction.csv'
TEST_IDENTITY = FILE_PATH / 'test_identity.csv'
TEST_TRANSACTION = FILE_PATH / 'test_transaction.csv'

SAVE_PREP_DATA_FOLDER = Path('../00_dataset/01_prep_data/')

In [14]:
# Query cruzamento
STMT_TEMPLATE = '''
SELECT 
    *
FROM read_csv('{tb_transaction}') AS t1
LEFT JOIN read_csv('{tb_identity}') AS t2
USING (TransactionID)
'''

In [21]:
train_query = STMT_TEMPLATE.format(tb_transaction=TRAIN_TRANSACTION, tb_identity=TRAIN_IDENTITY)
test_query = STMT_TEMPLATE.format(tb_transaction=TEST_TRANSACTION, tb_identity=TEST_IDENTITY)

In [22]:
print('Train - Dataset')
print(train_query)
print('-' * 30)
print('Test - Dataset')
print(test_query)

Train - Dataset

SELECT 
    *
FROM read_csv('..\00_dataset\00_raw_data\train_transaction.csv') AS t1
LEFT JOIN read_csv('..\00_dataset\00_raw_data\train_identity.csv') AS t2
USING (TransactionID)

------------------------------
Test - Dataset

SELECT 
    *
FROM read_csv('..\00_dataset\00_raw_data\test_transaction.csv') AS t1
LEFT JOIN read_csv('..\00_dataset\00_raw_data\test_identity.csv') AS t2
USING (TransactionID)



In [40]:
df_train = duckdb.sql(train_query) 
df_test = duckdb.sql(test_query)

In [41]:
df_train = df_train.pl()
df_test = df_test.pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [43]:
def norm_cols_name(col_name: str) -> str:
    if col_name == "isFraud":
        return "label"
    col_name = col_name.replace("-", "_").replace(" ", "_")
    col_name = re.sub(r'(?<=[a-z])(?=[A-Z])', '_', col_name)  # insere _ entre minuscula e maiuscula
    return col_name.lower()

In [46]:
map_cols_train = {col: norm_cols_name(col) for col in df_train.columns}
map_cols_test = {col: norm_cols_name(col) for col in df_test.columns}

In [ ]:
df_train = df_train.rename(map_cols_train)
df_test = df_test.rename(map_cols_test)

In [53]:
# salvando o dataframe
df_train.write_parquet(SAVE_PREP_DATA_FOLDER / 'df_train.pq')
df_test.write_parquet(SAVE_PREP_DATA_FOLDER / 'df_test.pq')

In [55]:
# validando se existe alguma diferença entre as colunas das duas tabelas
set(df_train.columns) - set(df_test.columns)

{'label'}